In [56]:
import sys
import os
import re

from langchain.tools import tool
from langchain.tools import Tool
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import initialize_agent
from langchain.agents import AgentType

sys.path.append(os.path.abspath(".."))

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [57]:
search_wikipedia.invoke("What is tool using?")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [ ]:
# Update your tools list to include the Wikipedia tool
from tools.simple_openai_tool import add_numbers
from langgraph.prebuilt import create_react_agent

tools_updated = [add_numbers, multiply_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_react_agent(
    model=llm,
    tools=tools_updated,
    prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

In [ ]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")

[]

Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: What is the population of Canada? Multiply it by 0.75
Name: None

--- Message 2 ---
Type: AIMessage
Content: 
Name: None
Tool calls: [{'name': 'search_wikipedia', 'args': {'query': 'Canada population'}, 'id': 'call_MhyLECZxel61agnBtviQG8GM', 'type': 'tool_call'}]

--- Message 3 ---
Type: ToolMessage
Content: Page: Population of Canada
Summary: Canada ranks 37th by population among countries of the world, comprising about 0.5% of the world's total, with about 41.5 million Canadians as of Q1 2026. Despite being the second-largest country by total area (fourth-largest by land area), the vast majority of the country is sparsely inhabited, with most of its population south of the 55th parallel north. Just over 60% of Canadians live in just two provinces: Ontario and Quebec. Though Canada's overall population density is low, many regions in the south, such as the Quebec City–Windsor Corridor, have population densities highe

In [ ]:
def calculate_power(input_text: str) -> dict:
    """
    Calculates the power of a number (x^y).

    Parameters:
    - input_text (str): A string like "2, 3", "2 3", "5^2", or "2 to the power of 3".

    Returns:
    - dict: {"result": <calculated value>} or an error message.
    """
    # Try to extract expressions like "5^2"
    match = re.search(r"(\d+(?:\.\d+)?)\s*\^+\s*(\d+(?:\.\d+)?)", input_text)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Try to extract expressions like "2 to the power of 3"
    match = re.search(r"(\d+(?:\.\d+)?)\s*(?:to\s+the\s+power\s+of)\s*(\d+(?:\.\d+)?)", input_text, re.IGNORECASE)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Fallback: assume two numbers separated by space or comma
    try:
        numbers = [float(num) for num in input_text.replace(",", " ").split()]
        if len(numbers) != 2:
            return {"result": "Invalid input. Please provide exactly two numbers."}
        base, exponent = numbers
        return {"result": base ** exponent}
    except ValueError:
        return {"result": "Invalid input format. Provide input like '2 3', '2^3', or '2 to the power of 3'."}

In [ ]:
power_tool = Tool(
   name="PowerTool",
   func=calculate_power,
   description="Calculates the power of a number (x^y). Input should be two numbers: base and exponent."
)

In [ ]:
tools_power= [power_tool]

agent_math= initialize_agent(
    tools_power,
    llm,
    agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose= True,
        handle_parsing_error= True
)

print(agent_math)


/var/folders/h1/rtnypb8d4td3v2lq5mjxz6qw0000gn/T/ipykernel_49510/3734409828.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [ ]:
agent_math.run("Calculate 5 to the power of 2.")



> Entering new AgentExecutor chain...
To calculate 5 to the power of 2, I will use the PowerTool to perform the calculation. 

Action: PowerTool  
Action Input: (5, 2)  
Observation: {'result': "Invalid input format. Provide input like '2 3', '2^3', or '2 to the power of 3'."}
Thought:It appears that the input format I used for the PowerTool was incorrect. I need to adjust my input to match the expected format. 

Thought: I will reformat the action input to use the correct syntax for the PowerTool. I'll express it as "5 to the power of 2".

Action: PowerTool  
Action Input: "5 to the power of 2"  
Observation: {'result': 25.0}
Thought:I now know the final answer.  
Final Answer: 25.0

> Finished chain.


'25.0'